# Cubic Pancake Parameter Space Exploration

Systematically explore the Cubic Pancake generator across all coset groups and subset values.

## Search Space
- **subset values**: 1–7 (each selects a different trio of prefix-reversal generators)
- **n values**: 4–16 for coset groups, 4–10 for full graph
- **coset types**: 5 groups (run separately, in parallel)

## Coset Groups
1. `different` — 2Different, 3Different, 4Different
2. `then` — Binary0then1, 0then1then2, etc.
3. `coincide` — 2Coincide, 3Coincide, etc.
4. `repeats` — Binary01Repeats, 012Repeats, etc.
5. `full_graph` — No coset (max_n=10, n! states)

## Cell 1: Imports and Setup

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'Utils'))

from explorer import Explorer, CUBIC_PANCAKE
from explorer.coset_groups import print_summary
import pandas as pd

## Cell 2: Explorer Setup

In [ ]:
exp = Explorer(
    config=CUBIC_PANCAKE,
    output_dir="results_cubic_pancake",
    min_n=4,
    max_n=16,
)
print(exp)
print_summary()

---
## Group 1: "different" (3 cosets)
- 2Different: [0, 1, 1, 1, ...]
- 3Different: [0, 1, 2, 2, ...]
- 4Different: [0, 1, 2, 3, 3, ...]

In [ ]:
%%time
df_different = exp.run_and_save(
    "different",
    parallel=True,
    max_workers=os.cpu_count(),
    max_n=16,
    subset_range=(1, 7),
    plot=True,
)
display(df_different.head(10))

---
## Group 2: "then" (4 cosets)
- Binary0then1: [0,0,...,1,1,...]
- 0then1then2: [0,...,1,...,2,...]
- 0then1then2then3: quarters
- 0then1then2then3then4: fifths

In [ ]:
%%time
df_then = exp.run_and_save(
    "then",
    parallel=True,
    max_workers=os.cpu_count(),
    max_n=16,
    subset_range=(1, 7),
    plot=True,
)
display(df_then.head(10))

---
## Group 3: "coincide" (5 cosets)
- 2Coincide: [..., x, x]
- 3Coincide: [..., x, x, x]
- 4Coincide, 5Coincide, 6Coincide

In [ ]:
%%time
df_coincide = exp.run_and_save(
    "coincide",
    parallel=True,
    max_workers=os.cpu_count(),
    max_n=16,
    subset_range=(1, 7),
    plot=True,
)
display(df_coincide.head(10))

---
## Group 4: "repeats" (4 cosets)
- Binary01Repeats: [0,1,0,1,...]
- Binary01Repeats_1: [0,1,0,1,...] with 1s at end
- 012Repeats: [0,1,2,0,1,2,...]
- 011Repeats: [0,1,1,0,1,1,...]

In [ ]:
%%time
df_repeats = exp.run_and_save(
    "repeats",
    parallel=True,
    max_workers=os.cpu_count(),
    max_n=16,
    subset_range=(1, 7),
    plot=True,
)
display(df_repeats.head(10))

---
## Group 5: "full_graph" (1 coset, max_n=10)
- FullGraph: No central state — explores the entire permutation group S_n
- **Warning**: |S_n| = n! grows very fast; keeping max_n=10

In [ ]:
%%time
df_full = exp.run_and_save(
    "full_graph",
    parallel=True,
    max_workers=os.cpu_count(),
    max_n=10,
    subset_range=(1, 7),
    plot=True,
)
display(df_full.head(10))

---
## Combined Analysis

Load all groups, concatenate, and plot diameter vs n faceted by coset group with subset as color.

In [ ]:
import plotly.express as px

dfs = []
for group in ["different", "then", "coincide", "repeats", "full_graph"]:
    try:
        df = exp.load_results(group)
        df["group"] = group
        dfs.append(df)
    except FileNotFoundError:
        print(f"No results found for group '{group}' — skipping.")

combined = pd.concat(dfs, ignore_index=True)
combined["subset"] = combined["subset"].astype(str)

fig = px.line(
    combined,
    x="n",
    y="diameter",
    color="subset",
    facet_col="group",
    facet_col_wrap=3,
    title="Cubic Pancake: Diameter by coset group and subset",
    labels={"n": "n", "diameter": "Diameter", "subset": "Subset"},
)
fig.update_layout(height=600)
fig.show()